# BIS API exploration

Probe the BIS MCP tools. The MCP server must be running (`python mcp_server/server.py`).

BIS needs no credentials. A dataflow (e.g. `WS_CBPOL`) is the dataset; its data
structure (often the same id with `WS_` → `BIS_`, e.g. `BIS_CBPOL`) declares the
dimensions and the codelists that decode them.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

from matplotlib import pyplot
from lib import config
from lib.utils import print_json_vertical
from utils import (
    call_tool,
    list_mcp_tools,
    show_dataflows,
    show_datastructure,
    get_series,
    decode_series,
    plot_bis_series,
)

pyplot.style.use(config.glyfish_style)

## Available tools

In [ ]:
await list_mcp_tools()

## Dataflows

The 29 published datasets — credit, policy rates, property prices, banking, etc.

In [ ]:
flows = await show_dataflows()

## Data structure

The dimensions that make up a series key, and the codelist decoding each.
Codelists are summarized by count (some hold 1000+ codes).

In [ ]:
dsd = await show_datastructure("BIS_TOTAL_CREDIT")

## Fetch and decode a series

Central bank policy rates for the US, UK, Japan and the euro area. `decode_series`
labels the dimension codes (`REF_AREA=US` → `United States`).

In [ ]:
data = await decode_series("WS_CBPOL", "M.US+GB+JP+XM", "BIS_CBPOL", start_period="2015-01")
for s in data["series"]:
    print(s["key"], "->", s["labels"])

## Plot a series

In [ ]:
us = next(s for s in data["series"] if s["dimensions"].get("REF_AREA") == "US")
plot_bis_series(us)

## Series keys

`key` is the dot-joined dimension values in DSD order. Omit a position to
wildcard it, use `+` for alternatives, or `all` for every series. Inspect the
data structure first to know the dimension order and valid codes.

In [ ]:
# every series in a flow (careful: large flows return a lot)
# data = await get_series("WS_CBPOL", "all", start_period="2026-01")
credit = await show_datastructure("BIS_TOTAL_CREDIT")